Instead of copying the information from these pages manually into a spreadsheet, I decided to learn something about pandas.

For what it's worth, I probably spent more time reading the documentation than I would have using `Ctrl-C + Ctrl-V`.

Practiced using regex to remove non alphanumeric characters and get everything formatted consistently - also an unnecessary effort, but it was interesting.

In [2]:
import pandas as pd

In [3]:
tables_grad = pd.read_html('https://campep.org/campeplstgrad.asp')
tables_residency = pd.read_html('https://campep.org/campeplstres.asp')

In [48]:
grad_programs = tables_grad[0].dropna()
residency_programs = tables_residency[0].drop(1, axis = 1).dropna()

# moving the column labels from the data to the header. this doesn't really matter because it exports the same either way...
grad_programs.columns = grad_programs.iloc[0]
# dropping headers and the accreditation column
grad_programs = grad_programs.drop(1).drop('Initial Accreditation', axis = 1)

# repeat for residency programs
residency_programs.columns = residency_programs.iloc[0]
residency_programs = residency_programs.drop([0, 137]).drop('Initial Accreditation', axis = 1)

In [49]:
grad_programs['Institution'] = grad_programs['Institution'].str.replace(r'[/\-()]', ' ', regex = True) # replace punctuation characters with spaces
grad_programs['Institution'] = grad_programs['Institution'].str.replace(r'[^\w\s]', '', regex = True) # remove special characters
grad_programs['Institution'] = grad_programs['Institution'].str.replace(' +', ' ', regex = True) # removing multiple spaces.
grad_programs['Institution'] = grad_programs['Institution'].str.replace('é', 'e') # battling the despicable french

# repeating for residencies
residency_programs['Institution'] = residency_programs['Institution'].str.replace(r'[/\-()]', ' ', regex = True)
residency_programs['Institution'] = residency_programs['Institution'].str.replace(r'[^\w\s]', '', regex = True)
residency_programs['Institution'] = residency_programs['Institution'].str.replace(' +', ' ', regex = True)
residency_programs['Institution'] = residency_programs['Institution'].str.replace('é', 'e')

In [50]:
# removing duplicates from the list of residency programs.
# many universities have both therapy and imaging programs.
grad_programs.drop_duplicates(inplace = True, ignore_index = True)
residency_programs.drop_duplicates(inplace = True, ignore_index = True)

In [51]:
# adding additional columns for my personal use
grad_programs['Date Checked'] = None
grad_programs['Notes'] = None
residency_programs['Date Checked'] = None
residency_programs['Notes'] = None

In [52]:
# Creating a table of all institutions which have residency programs but no grad program.
additional_programs = pd.merge(grad_programs, residency_programs, how = 'outer', indicator = True).query("_merge == 'right_only'").drop("_merge", axis = 1)

In [53]:
grad_programs.to_csv("grad_programs.csv", index = False)
residency_programs.to_csv("residency_programs.csv", index = False)
additional_programs.to_csv("additional_programs.csv", index = False)